# Day 04: Multi-Sensor Fusion & Localization Capstone
**State Estimation and Localization for Self-Driving Cars**

Welcome to the capstone session! Today we build the industry-standard vehicle localization engine:
1. **The Error-State Extended Kalman Filter (ES-EKF)**:
   - Why Error-State? Decoupling continuous nominal kinematics from 3D error quaternion perturbation vector $\delta \boldsymbol{\theta}$.
   - High-rate 100 Hz IMU nominal state integration.
   - Low-rate 10 Hz GNSS position corrections.
2. **Sensor Fusion Pipeline**:
   - Covariance propagation and Joseph-form measurement updates.
   - Error injection and reset mechanism.
3. **Challenging GNSS Tunnel Outage Scenario**:
   - 10-second complete GNSS signal blackout.
   - Analysis of covariance envelope explosion and instant re-convergence upon satellite re-acquisition.
4. **Statistical Verification Metrics**:
   - Normalized Estimation Error Squared (NEES) for state consistency.
   - Normalized Innovation Squared (NIS) for filter health monitoring.


In [ ]:
import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from position_class import (
    ErrorStateEKF,
    Quaternion,
    skew_symmetric
)

print("Day 04 Capstone Environment Ready!")


## 1. The Error-State EKF (ES-EKF) Architecture

The state is decomposed into:
- **Nominal State** (large, non-linear): $\mathbf{x} = [\mathbf{p}, \mathbf{v}, \mathbf{q}]$
- **Error State** (small, linear): $\delta \mathbf{x} = [\delta \mathbf{p}, \delta \mathbf{v}, \delta \boldsymbol{\theta}]^T \in \mathbb{R}^9$

Measurement Model for GNSS:

$$\mathbf{y}_k = \mathbf{p}_{\text{GNSS}} - \mathbf{p}_{\text{nom}} = \mathbf{H} \delta \mathbf{x}_k + \mathbf{v}_k, \quad \mathbf{H} = \begin{bmatrix} \mathbf{I}_{3 \times 3} & \mathbf{0}_{3 \times 3} & \mathbf{0}_{3 \times 3} \end{bmatrix}$$


In [ ]:
# Simulation of 90-Second Vehicle Loop with 10-Second Tunnel Outage
np.random.seed(42)
dt = 0.01  # 100 Hz IMU
total_time = 60.0
steps = int(total_time / dt)
t = np.linspace(0, total_time, steps)

# 1. Generate Ground Truth Race Track Loop
R = 50.0
speed = 15.0  # m/s
omega = speed / R

gt_x = R * np.sin(omega * t)
gt_y = R * np.cos(omega * t) - R
gt_z = np.zeros_like(t)

gt_vx = R * omega * np.cos(omega * t)
gt_vy = -R * omega * np.sin(omega * t)
gt_vz = np.zeros_like(t)

# Ground truth IMU readings
acc_centripetal = (speed ** 2) / R
gt_ax = -acc_centripetal * np.sin(omega * t)
gt_ay = -acc_centripetal * np.cos(omega * t)
gt_az = np.full_like(t, 9.81)  # Specific force counteracting gravity

# Initialize ES-EKF
init_p = np.array([gt_x[0], gt_y[0], 0.0])
init_v = np.array([gt_vx[0], gt_vy[0], 0.0])
init_q = Quaternion([1.0, 0.0, 0.0, 0.0])
init_cov = np.diag([1.0, 1.0, 1.0, 0.1, 0.1, 0.1, 0.01, 0.01, 0.01])

es_ekf = ErrorStateEKF(init_pos=init_p, init_vel=init_v, init_quat=init_q, init_cov=init_cov, accel_noise_std=0.2, gyro_noise_std=0.02)

# GPS Configuration: 10 Hz rate, 10-second tunnel outage between t = 25s and t = 35s
gps_dt = 0.1
gps_subsample = int(gps_dt / dt)
r_gps = np.eye(3) * (0.8 ** 2)

est_pos, est_cov_pos = [], []
nees_list, nis_list, nis_times = [], [], []

for k in range(steps):
    current_time = t[k]
    
    # 1. Noisy IMU Measurement
    f_b = np.array([0.0, acc_centripetal, 9.81]) + np.random.normal(0, 0.15, 3)
    omega_b = np.array([0.0, 0.0, -omega]) + np.random.normal(0, 0.01, 3)
    
    p, v, q, P = es_ekf.predict(f_b, omega_b, dt)
    
    # 2. GNSS Correction (10 Hz, inactive in tunnel t in [25, 35])
    is_in_tunnel = 25.0 <= current_time <= 35.0
    if k % gps_subsample == 0 and not is_in_tunnel:
        gps_meas = np.array([gt_x[k], gt_y[k], gt_z[k]]) + np.random.normal(0, 0.8, 3)
        p, v, q, P, nis = es_ekf.update_gnss_position(gps_meas, r_gps)
        nis_list.append(nis)
        nis_times.append(current_time)
        
    est_pos.append(p)
    est_cov_pos.append(np.diag(P)[0:3])
    
    # Position error NEES
    err_p = p - np.array([gt_x[k], gt_y[k], gt_z[k]])
    nees = float(err_p.T @ np.linalg.inv(P[0:3, 0:3]) @ err_p)
    nees_list.append(nees)

est_pos = np.array(est_pos)
est_cov_pos = np.array(est_cov_pos)
pos_3sigma = 3.0 * np.sqrt(est_cov_pos)

print("Simulation Completed!")


## 2. Interactive Performance & Tunnel Outage Visualizations

Observe how:
1. When GNSS is available, position error remains bounded within centimeters.
2. During the 10-second tunnel outage ($t \in [25\text{s}, 35\text{s}]$), the $3\sigma$ covariance bounds expand naturally, capturing uncertainty.
3. Upon exiting the tunnel, the first GNSS fix immediately shrinks the error covariance back down.


In [ ]:
# Plot 1: 2D Multi-Sensor Trajectory & Tunnel Region
fig_traj = go.Figure()
fig_traj.add_trace(go.Scatter(x=gt_x, y=gt_y, mode='lines', name='Ground Truth Path', line=dict(color='black', width=3)))
fig_traj.add_trace(go.Scatter(x=est_pos[:, 0], y=est_pos[:, 1], mode='lines', name='ES-EKF Fused Trajectory', line=dict(color='blue', width=2)))

# Highlight Tunnel Zone
tunnel_mask = (t >= 25.0) & (t <= 35.0)
fig_traj.add_trace(go.Scatter(
    x=est_pos[tunnel_mask, 0], y=est_pos[tunnel_mask, 1],
    mode='lines', name='Tunnel Dead Reckoning (No GNSS)',
    line=dict(color='red', width=3, dash='dash')
))

fig_traj.update_layout(
    title='<b>ES-EKF Multi-Sensor Vehicle Trajectory (IMU + GNSS + 10s Tunnel Outage)</b>',
    xaxis_title='X [m] (East)', yaxis_title='Y [m] (North)',
    height=550
)
fig_traj.show()


In [ ]:
# Plot 2: Position Error vs 3-Sigma Bounds & Consistency Metrics
fig_metrics = make_subplots(
    rows=2, cols=1,
    subplot_titles=("<b>X-Position Error with 3σ Covariance Bounds (Tunnel Outage Highlighted)</b>", "<b>Normalized Estimation Error Squared (NEES)</b>")
)

x_err = est_pos[:, 0] - gt_x
fig_metrics.add_trace(go.Scatter(x=t, y=x_err, mode='lines', name='Position Error X (m)', line=dict(color='blue')), row=1, col=1)
fig_metrics.add_trace(go.Scatter(x=t, y=pos_3sigma[:, 0], mode='lines', name='+3σ Bound', line=dict(color='gray', dash='dash')), row=1, col=1)
fig_metrics.add_trace(go.Scatter(x=t, y=-pos_3sigma[:, 0], mode='lines', name='-3σ Bound', line=dict(color='gray', dash='dash'), fill='tonexty', fillcolor='rgba(180,180,180,0.2)'), row=1, col=1)

# Add tunnel shaded rect
fig_metrics.add_vrect(x0=25.0, x1=35.0, fillcolor="red", opacity=0.15, line_width=0, annotation_text="10s Tunnel Outage", row=1, col=1)

# NEES
fig_metrics.add_trace(go.Scatter(x=t, y=nees_list, mode='lines', name='Position NEES', line=dict(color='purple')), row=2, col=1)
fig_metrics.add_hline(y=7.81, line_dash="dash", line_color="red", annotation_text="95% Upper Chi-Square Bound (df=3)", row=2, col=1)

fig_metrics.update_layout(height=750, title_text="<b>Day 04: ES-EKF Consistency & Tunnel Outage Resilience Analysis</b>")
fig_metrics.show()
